# 03 — SHAP analysis
Global and local explanations for the trained PD model artifact.

If `artifacts/model.pkl` is missing, run notebook 02 first (or `make local-pipeline`).


In [ ]:
%pip install lightgbm shap loguru scikit-learn matplotlib catboost xgboost numpy pandas scipy

import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import ARTIFACTS_DIR
from src.models.scoring.artifacts import ScoringArtifact
from src.models.scoring.explain import compute_shap_values, get_top_reasons

artifact_path = ARTIFACTS_DIR / "model.pkl"
if not artifact_path.exists():
    print("No artifact found — training a small model first...")
    from scripts.run_local_pipeline import run_pandas_pipeline
    run_pandas_pipeline(n=2500)

art = ScoringArtifact.load(ARTIFACTS_DIR)
print(art.model_type, art.metrics)

In [ ]:
rng = np.random.default_rng(0)
X = pd.DataFrame({
    f: rng.normal(art.feature_medians.get(f, 0.0), 0.1, 200)
    for f in art.feature_names
})
shap_values, explainer = compute_shap_values(art.model, X)
shap_values.shape

In [ ]:
import shap
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.show()

In [ ]:
row = shap_values[0]
print(get_top_reasons(row, art.feature_names, top_n=5))